# 5. Entrenamiento, ajuste y validación

**Corrida preliminar con datos reales.** Se comparan Ridge, Elastic Net, SVR-RBF, árbol,
bosque aleatorio y proceso gaussiano, además del baseline. La red neuronal sigue como extensión
exploratoria fuera de esta comparación.

En cada fold externo, la CV interna selecciona hiperparámetros y familia por MAE. Después se
reajusta usando su entrenamiento externo y se predicen los casos reservados. `selected` representa
ese procedimiento completo: puede elegir una familia distinta en cada fold.

**Para repetir:** ejecutar todas las celdas. Si existe una corrida completa de los mismos datos y
configuración se verifica y reutiliza. Cambiar `FORZAR_NUEVA_EJECUCION` a `True` inicia una nueva
corrida fechada. También puede ejecutarse `python -m src.experimento --run --fit-final` desde la raíz.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src' / 'experimento.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Abra el notebook desde este repositorio.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.experimento import read_config, inspect_plan, load_model_data, find_completed_run, run_experiment
from src.particiones import nested_plan
from src.modelo_baseline import build_baseline, evaluate_regression
from dataclasses import asdict

config, config_hash = read_config(ROOT / 'config' / 'model_config.yml')
X, y, trace, groups, quality, data_path, report_path = load_model_data(config)
pd.set_option('display.float_format', lambda value: f'{value:.3f}')
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
print('Datos:', data_path.relative_to(ROOT))
print('Objetivo:', y.name, '[N]')

Datos: data\processed\preliminary\model_table.csv
Objetivo: First_Failure_Load_N [N]


## 5.1 Comprobar preparación

In [2]:
status = inspect_plan(config)
assert status['status'] == 'ready_for_reviewed_execution'
display(pd.DataFrame({'familia': list(status['candidates']), 'estado': 'lista para CV anidada'}))
print('Protocolo:', config['protocol']['outer_splits'], 'folds externos ×', config['protocol']['inner_splits'], 'internos')

,familia,estado
0,ridge,lista para CV anidada
1,elastic_net,lista para CV anidada
2,support_vector,lista para CV anidada
3,decision_tree,lista para CV anidada
4,random_forest,lista para CV anidada
5,gaussian_process,lista para CV anidada


Protocolo: 5 folds externos × 3 internos


## 5.2 Ejecutar o recuperar una corrida íntegra

In [3]:
FORZAR_NUEVA_EJECUCION = False
run_dir = None if FORZAR_NUEVA_EJECUCION else find_completed_run(config, config_hash, require_final=True)
if run_dir is None:
    print('Entrenando seis familias, baseline y reajuste final...')
    run_dir = run_experiment(config, config_hash, fit_final=True)
    print('Nueva corrida completada.')
else:
    print('Corrida real recuperada; hashes de datos, configuración y artefactos verificados.')
print('Ejecución:', run_dir.relative_to(ROOT))
evaluation = json.loads((run_dir / 'evaluation.json').read_text(encoding='utf-8'))
manifest = json.loads((run_dir / 'experiment_manifest.json').read_text(encoding='utf-8'))
print('Estado:', manifest['status'], '| Finalizada:', manifest['completed_at_utc'])

Corrida real recuperada; hashes de datos, configuración y artefactos verificados.
Ejecución: results\20261003T021921587329Z
Estado: complete | Finalizada: 2026-10-03T02:19:47.251299+00:00


## 5.3 Modelos seleccionados y comparación fuera de muestra

In [4]:
selections = pd.DataFrame([{'fold': item['fold'], 'familia elegida por CV interna': item['selected_by_inner_cv']}
                           for item in evaluation['selection_by_fold']])
display(selections)
metrics = pd.DataFrame(evaluation['pooled_oof_metrics']).T.rename_axis('procedimiento')
display(metrics)
print('MAE, RMSE y MedAE: N. R²: sin unidad.')
print('Las métricas por familia son diagnósticas; no se usa el mejor error externo para seleccionar.')

,fold,familia elegida por CV interna
0,1,elastic_net
1,2,decision_tree
2,3,decision_tree
3,4,decision_tree
4,5,elastic_net


,mae,rmse,medae,r2
procedimiento,,,,
baseline,51.558,66.803,38.770,-0.293
ridge,12.036,15.547,9.112,0.930
elastic_net,11.490,15.048,6.946,0.934
support_vector,16.183,20.975,11.356,0.873
decision_tree,11.420,15.280,8.608,0.932
random_forest,15.730,20.517,12.054,0.878
gaussian_process,18.500,21.854,16.540,0.862
selected,9.671,12.982,7.047,0.951


MAE, RMSE y MedAE: N. R²: sin unidad.
Las métricas por familia son diagnósticas; no se usa el mejor error externo para seleccionar.


## 5.4 Reajuste para uso posterior y advertencias

In [5]:
final_info = json.loads((run_dir / 'final_model_info.json').read_text(encoding='utf-8'))
display(pd.DataFrame([
    {'artefacto': 'final_model.joblib', 'descripción': 'pipeline reajustado con todos los datos', 'familia': final_info['family']},
    {'artefacto': 'oof_predictions.csv', 'descripción': 'predicciones externas para calcular métricas', 'familia': 'todos los procedimientos'},
    {'artefacto': 'experiment_manifest.json', 'descripción': 'versiones y hashes de la ejecución', 'familia': '—'},
]))
print(final_info['note'])
print('Advertencias durante CV externa:', len(evaluation['warnings']))
print('Advertencias del reajuste final:', len(final_info['warnings']))
if evaluation['warnings']:
    display(pd.DataFrame(evaluation['warnings']).drop_duplicates())
if final_info['warnings']:
    display(pd.DataFrame({'advertencia': final_info['warnings']}))

,artefacto,descripción,familia
0,final_model.joblib,pipeline reajustado con todos los datos,decision_tree
1,oof_predictions.csv,predicciones externas para calcular métricas,todos los procedimientos
2,experiment_manifest.json,versiones y hashes de la ejecución,—


Reajuste con todos los datos; no tiene una nueva evaluación independiente
Advertencias durante CV externa: 0
Advertencias del reajuste final: 0


El pipeline reajustado se selecciona por CV sobre todos los datos y sirve como artefacto preliminar
para inferencia. Sus predicciones sobre datos de entrenamiento no constituyen una evaluación nueva.
Las métricas principales pertenecen al procedimiento `selected` fuera de muestra.

Siguiente: [06 · Evaluación e interpretabilidad](06_Evaluacion_Final_e_Interpretabilidad.ipynb).